# Task 2: RAG from scratch (embeddings + NumPy/FAISS, no LangChain/LlamaIndex)
Document: **NIST AI Risk Management Framework 1.0** (public, about 48 pages).
Runs on CPU or GPU. Fill in your 15 questions in section 3 before running the evaluation.

In [ ]:
!pip install -q sentence-transformers faiss-cpu pypdf requests anthropic

In [ ]:
import re, io, requests, numpy as np, faiss
from collections import Counter
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

URL = "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf"
r = requests.get(URL, timeout=60); r.raise_for_status()
reader = PdfReader(io.BytesIO(r.content))
pages = [re.sub(r"\s+", " ", p.extract_text() or "").strip() for p in reader.pages]
print("pages:", len(pages), "| words:", sum(len(p.split()) for p in pages))
# If the download fails, upload any >=30 page PDF in the Files panel and use PdfReader("yourfile.pdf").

pages: 48 | words: 15897


## 1. Chunking, embedding, FAISS index

In [ ]:
def norm(s): return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def build_chunks(size, overlap=0.2):
    words, wpage = [], []
    for pno, txt in enumerate(pages, 1):
        w = txt.split(); words += w; wpage += [pno] * len(w)
    step = max(1, int(size * (1 - overlap))); chunks = []
    for s in range(0, len(words), step):
        piece = words[s:s + size]
        if len(piece) < size * 0.3 and chunks: break
        text = " ".join(piece)
        chunks.append({"text": text, "page": wpage[s], "norm": norm(text)})
    return chunks

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")   # max 512 tokens, so keep chunks <= ~350 words
QPREFIX = "Represent this sentence for searching relevant passages: "

def tokens(s): return re.findall(r"[a-z0-9]+", s.lower())

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.tf = [Counter(tokens(d)) for d in docs]
        self.len = np.array([sum(c.values()) for c in self.tf]); self.avg = self.len.mean()
        df = Counter()
        for c in self.tf: df.update(c.keys())
        N = len(docs); self.idf = {t: np.log(1 + (N - n + .5) / (n + .5)) for t, n in df.items()}
    def scores(self, q):
        s = np.zeros(len(self.tf))
        for t in set(tokens(q)):
            if t not in self.idf: continue
            for i, c in enumerate(self.tf):
                f = c.get(t, 0)
                if f: s[i] += self.idf[t] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return s

class Index:
    def __init__(self, size):
        self.size = size; self.chunks = build_chunks(size)
        emb = embedder.encode([c["text"] for c in self.chunks], normalize_embeddings=True,
                              batch_size=32, show_progress_bar=False).astype("float32")
        self.faiss = faiss.IndexFlatIP(emb.shape[1]); self.faiss.add(emb)
        self.bm25 = BM25([c["text"] for c in self.chunks])
        self._ce = None
    def dense(self, q, k):
        qe = embedder.encode([QPREFIX + q], normalize_embeddings=True).astype("float32")
        return list(self.faiss.search(qe, k)[1][0])
    def sparse(self, q, k): return list(np.argsort(-self.bm25.scores(q))[:k])
    def hybrid(self, q, k, c=60):                      # reciprocal rank fusion
        score = Counter()
        for lst in (self.dense(q, 50), self.sparse(q, 50)):
            for rank, i in enumerate(lst): score[i] += 1 / (c + rank + 1)
        return [i for i, _ in score.most_common(k)]
    def rerank(self, q, k, pool=20):                   # dense top-20 -> cross-encoder -> top-k
        if self._ce is None: self._ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        cand = self.hybrid(q, pool)
        sc = self._ce.predict([(q, self.chunks[i]["text"]) for i in cand])
        return [cand[j] for j in np.argsort(-sc)[:k]]

SIZES = [100, 300]            # words per chunk (about 130 and 400 tokens)
indexes = {s: Index(s) for s in SIZES}
for s, ix in indexes.items(): print(f"chunk size {s} words -> {len(ix.chunks)} chunks")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

chunk size 100 words -> 199 chunks
chunk size 300 words -> 66 chunks


## 2. Helper to write your questions
Use `find("some phrase")` to locate passages while reading the PDF, and `show_page(n)` to read a page.

In [ ]:
def find(phrase):
    p = norm(phrase)
    for i, t in enumerate(pages, 1):
        j = norm(t).find(p)
        if j >= 0: print(f"page {i}: ...{norm(t)[max(0,j-80):j+len(p)+80]}...")
def show_page(n): print(pages[n - 1])

## 3. The 15 test questions
These are drafts. Read each passage in the PDF, reword the questions in your own style, and replace any you don't understand. (`page` = printed page number in the PDF, which is lower than the PDF viewer's page number.)

For each: a question, plus a **gold** = a distinctive 8-15 word phrase copied from the passage that answers it
(inside one page, avoiding words broken by a hyphen at a line end). Retrieval counts as a hit if a retrieved chunk
contains that phrase. Mix easy (keyword overlap) and hard (paraphrased question, or answer spread across a sentence).
Replace every TODO. `check_gold()` tells you which phrases it cannot find.

In [ ]:
QUESTIONS = [
    {"question": "What are the four functions that make up the AI RMF Core?", "gold": "the Core is composed of four functions: GOVERN, MAP, MEASURE, and MANAGE", "page": 20},
    {"question": "How does the framework define risk?", "gold": "risk refers to the composite measure of an event", "page": 4},
    {"question": "Does the framework tell organizations how much risk they should be willing to accept?", "gold": "While the AI RMF can be used to prioritize risk, it does not prescribe risk tolerance", "page": 7},
    {"question": "How does the document distinguish resilience from security?", "gold": "resilience is the ability to return to normal function after an unexpected adverse event", "page": 15},
    {"question": "Which categories of bias does NIST say need to be managed?", "gold": "three major categories of AI bias to be considered and managed: systemic, computational and statistical", "page": 18},
    {"question": "What is the Playbook that accompanies the framework for?", "gold": "the NIST AI RMF Playbook, is available to help organizations navigate the AI RMF", "page": 21},
    {"question": "What should happen if an AI system poses unacceptable risk?", "gold": "development and deployment should cease in a safe manner until risks", "page": 8},
    {"question": "Which function sets up the context for framing an AI system's risks?", "gold": "The MAP function establishes the context to frame risks related to an AI system", "page": 24},
    {"question": "Once governance is in place, which function do most users tackle first?", "gold": "After instituting the outcomes in GOVERN, most users of the AI RMF would start with the MAP function", "page": 21},
    {"question": "What can an organization do about a high-priority AI risk?", "gold": "Risk response options can include mitigating, transferring, avoiding, or accepting", "page": 32},
    {"question": "Is it a good goal to remove all negative risk from an AI system?", "gold": "Attempting to eliminate negative risk entirely can be counterproductive in practice", "page": 7},
    {"question": "Which testing and validation processes does NIST's modified lifecycle figure emphasise?", "gold": "(TEVV) processes throughout an AI lifecycle", "page": 9},
    {"question": "Why might AI systems need maintenance more often than traditional software?", "gold": "AI systems may require more frequent maintenance and triggers for conducting corrective maintenance due to data, model, or concept drift", "page": 38},
    {"question": "What does a Target Profile describe?", "gold": "A Target Profile indicates the outcomes needed to achieve the desired or target AI risk management goals", "page": 33},
    {"question": "Which GOVERN subcategory requires keeping a list of AI systems?", "gold": "Mechanisms are in place to inventory AI systems and are resourced according to organizational risk priorities", "page": 23},
]

def check_gold():
    assert len(QUESTIONS) >= 15, f"you have {len(QUESTIONS)} questions, need 15"
    full = norm(" ".join(pages)); bad = [q["question"] for q in QUESTIONS if norm(q["gold"]) not in full]
    print("all gold phrases found" if not bad else f"NOT FOUND in document: {bad}")
check_gold()

NOT FOUND in document: ['How does the document distinguish resilience from security?', 'What can an organization do about a high-priority AI risk?', 'Why might AI systems need maintenance more often than traditional software?']


## 4. Retrieval accuracy for both chunk sizes

In [ ]:
def evaluate(ix, method, ks=(1, 3, 5)):
    hits = {k: 0 for k in ks}; misses = []
    for q in QUESTIONS:
        got = method(q["question"], max(ks)); g = norm(q["gold"])
        rank = next((r for r, i in enumerate(got, 1) if g in ix.chunks[i]["norm"]), None)
        for k in ks: hits[k] += rank is not None and rank <= k
        if rank is None or rank > 5: misses.append((q, got))
    return {f"recall@{k}": round(hits[k] / len(QUESTIONS), 3) for k in ks}, misses

rows = []
for s, ix in indexes.items():
    for name in ["dense", "sparse", "hybrid", "rerank"]:
        res, _ = evaluate(ix, getattr(ix, name)); rows.append({"chunk_words": s, "method": name, **res})
import pandas as pd
print(pd.DataFrame(rows).to_string(index=False))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

 chunk_words method  recall@1  recall@3  recall@5
         100  dense     0.333     0.533     0.600
         100 sparse     0.333     0.467     0.467
         100 hybrid     0.333     0.467     0.600
         100 rerank     0.467     0.467     0.533
         300  dense     0.267     0.533     0.600
         300 sparse     0.333     0.533     0.533
         300 hybrid     0.333     0.533     0.667
         300 rerank     0.400     0.800     0.800


**Interpretation guide for your write-up.**
- Small chunks: precise, less noise per chunk, but a passage's context may be cut across chunks (lower recall for questions needing context).
- Large chunks: more likely to *contain* the answer (recall can look higher, which is partly an artefact of the metric),
  but the embedding becomes a blurry average of several topics and you pass more tokens to the LLM.
- Say which was better on YOUR numbers and why, and mention the trade-off in LLM context cost.

## 5. Failure cases (pick 2 for your report)

In [ ]:
ix = indexes[SIZES[0]]
_, misses = evaluate(ix, ix.dense)
for q, got in misses[:5]:
    print("Q:", q["question"], "| gold:", q["gold"])
    for i in got[:3]: print(f"   p{ix.chunks[i]['page']}: {ix.chunks[i]['text'][:200]}...")
    print()
# Compare: does ix.sparse / ix.hybrid / ix.rerank fix the same question? That's your "what would fix it" evidence.

Q: What are the four functions that make up the AI RMF Core? | gold: the Core is composed of four functions: GOVERN, MAP, MEASURE, and MANAGE
   p24: and across organizations about risks, decision- making processes, responsibilities, common pitfalls, TEVV practices, and approaches for continuous improvement; • greater contextual knowledge for incre...
   p8: describes four specific functions to help organizations address the risks of AI systems in practice. These functions – GOVERN, MAP, MEASURE, and MANAGE – are broken down further into categories and su...
   p4: Part 2: Core and Profiles 20 5 AI RMF Core 20 5.1 Govern 21 5.2 Map 24 5.3 Measure 28 5.4 Manage 31 6 AI RMF Profiles 33 Appendix A: Descriptions of AI Actor Tasks from Figures 2 and 3 35 Appendix B: ...

Q: How does the document distinguish resilience from security? | gold: resilience is the ability to return to normal function after an unexpected adverse event
   p20: relate to adversarial examples, data poisoning, and the

## 6. Final answer with an LLM (any LLM works; this uses the Claude API)
Add your key in Colab: left sidebar -> key icon -> Secrets -> `ANTHROPIC_API_KEY`.
Free alternative: swap `ask_llm` for a local Ollama / Hugging Face model.

In [ ]:
from google import genai
from google.colab import userdata
import os

def get_api_key():
    # Try to find the key under various common names
    for key_name in ['GOOGLE_API_KEY', 'GEMINI_API_KEY', 'gemini_api_key', 'Anthropic_API_Key', 'ANTHROPIC_API_KEY']:
        try:
            val = userdata.get(key_name)
            if val:
                print(f"Successfully loaded API key from secret: '{key_name}'")
                return val
        except Exception:
            continue
    return None

api_key = get_api_key()
if not api_key:
    raise ValueError("Could not find any API key in Colab Secrets. Please open the left sidebar, click the '🔑' (Secrets) icon, and add a secret named 'GOOGLE_API_KEY' with your Gemini API key, then enable 'Notebook access'.")

client = genai.Client(api_key=api_key)

def ask_llm(prompt):
    try:
        response = client.models.generate_content(
            model='gemini-3.8-flash',
            contents=prompt,
        )
        return response.text
    except Exception as e:
        return f"[Error calling Gemini API: {e}. Please make sure your API key has correct permissions.]"

def answer(question, ix=indexes[300], k=5):
    ctx = [ix.chunks[i] for i in ix.rerank(question, k)]
    context = "\n\n".join(f"[page {c['page']}] {c['text']}" for c in ctx)
    return ask_llm(f"Answer using ONLY the context. Cite page numbers. If not in the context, say so.\n\n"
                   f"Context:\n{context}\n\nQuestion: {question}")

print(answer(QUESTIONS[0]["question"]))

Successfully loaded API key from secret: 'GEMINI_API_KEY'
[Error calling Gemini API: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. Please make sure your API key has correct permissions.]


### Running the RAG Pipeline on All 15 Questions
Let's run our hybrid-rerank RAG model over all 15 questions to generate answers using Gemini, compare with the gold phrases, and display the results cleanly.

In [ ]:
import pandas as pd
import time

# Let's use the best performing index configuration (300 words with rerank)
best_index = indexes[300]
results = []

for idx, q_info in enumerate(QUESTIONS, 1):
    question = q_info["question"]
    gold = q_info["gold"]
    expected_page = q_info["page"]

    print(f"Processing Q{idx}/{len(QUESTIONS)}: {question[:50]}...")

    # Retrieve top 5 most relevant chunks using rerank
    retrieved_indices = best_index.rerank(question, k=5)
    retrieved_chunks = [best_index.chunks[i] for i in retrieved_indices]

    # Build context and tracking retrieved pages
    context_blocks = []
    retrieved_pages = []
    for c in retrieved_chunks:
        context_blocks.append(f"[page {c['page']}] {c['text']}")
        retrieved_pages.append(c['page'])

    context_str = "\n\n".join(context_blocks)

    # Check if the gold answer phrase exists in any of our retrieved chunks
    g_norm = norm(gold)
    found_gold = any(g_norm in c["norm"] for c in retrieved_chunks)

    # Ask LLM
    llm_prompt = (
        f"Answer the following question using ONLY the provided context. "
        f"Always cite the page number(s) you found the answer in inside brackets like [page X].\n\n"
        f"Context:\n{context_str}\n\n"
        f"Question: {question}"
    )

    try:
        llm_answer = ask_llm(llm_prompt)
    except Exception as e:
        llm_answer = f"[Error generating response: {e}]"

    results.append({
        "Index": idx,
        "Question": question,
        "Expected Page": expected_page,
        "Retrieved Pages": list(set(retrieved_pages)),
        "Gold Phrase Found": "Yes" if found_gold else "No",
        "Answer": llm_answer
    })

    # Brief sleep to avoid rapid sequential requests
    time.sleep(1.0)

# Convert to DataFrame for a beautiful visual layout
df_results = pd.DataFrame(results)
print("Done processing all questions!")

In [ ]:
# Display the evaluation summary
display(df_results[["Index", "Question", "Expected Page", "Retrieved Pages", "Gold Phrase Found"]])

,Index,Question,Expected Page,Retrieved Pages,Gold Phrase Found
0,1,What are the four functions that make up the A...,20,"[24, 25, 37, 7]",Yes
1,2,How does the framework define risk?,4,"[8, 9, 11, 7]",Yes
2,3,Does the framework tell organizations how much...,7,"[7, 11, 12, 13, 47]",Yes
3,4,How does the document distinguish resilience f...,15,"[34, 7, 43, 20, 21]",No
4,5,Which categories of bias does NIST say need to...,18,"[36, 37, 7, 22, 23]",Yes
5,6,What is the Playbook that accompanies the fram...,21,"[1, 7, 25, 26, 27]",Yes
6,7,What should happen if an AI system poses unacc...,8,"[9, 10, 19, 13]",Yes
7,8,Which function sets up the context for framing...,24,"[36, 29, 30, 7]",Yes
8,9,"Once governance is in place, which function do...",21,"[25, 26, 27, 5]",Yes
9,10,What can an organization do about a high-prior...,32,"[5, 9, 12, 13, 25]",No


In [ ]:
# Let's print each question and the beautifully detailed LLM response with its source citations
for item in results:
    print("="*80)
    print(f"Q{item['Index']}: {item['Question']}")
    print(f"Gold Found in Retrieval: {item['Gold Phrase Found']}")
    print("-"*80)
    print(item["Answer"])
    print("\n")

Q1: What are the four functions that make up the AI RMF Core?
Gold Found in Retrieval: Yes
--------------------------------------------------------------------------------
[Error calling Gemini API: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. Please make sure your API key has correct permissions.]


Q2: How does the framework define risk?
Gold Found in Retrieval: Yes
--------------------------------------------------------------------------------
[Error calling Gemini API: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. Please make sure your API key has correct permissions.]


Q3: Does the framework tell organizations how much risk they should be willing to accept?
Gold Found in Retrieval: Y